# SSN Builder — sequence-based (mmseqs)

Builds **Sequence Similarity Networks (SSN)** from a FASTA file using mmseqs2.  
For structure-based networks (foldseek / PDB), see `build_stsn.ipynb`.

Run cells top to bottom, skipping optional ones as needed.

## Setup

In [2]:
import sys, logging
from pathlib import Path

SRC_DIR = Path("../src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("ssn")

In [3]:
# Shared paths — edit these before running anything else
INPUT_FASTA = Path("../data/GH43_0.8-0.8_catdom.fa")
PREFIX      = "test_run"
OUTPUT_DIR  = Path("../results/GH43_test")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

---
## Step 1 — Redundancy reduction *(optional)*

Run **one** of the two cells below.

### 1a · Cluster with mmseqs

In [3]:
from mmseqs import easy_cluster

result = easy_cluster(
    fasta      = INPUT_FASTA,
    output_dir = OUTPUT_DIR / "cluster",
    prefix     = PREFIX,
    min_seq_id = 0.8,
    coverage   = 0.8,
    cov_mode   = 0,    # 0 = bidirectional, 1 = query, 2 = target
    log        = log,
)
rep_fasta   = result["rep_seq_fasta"]
cluster_tsv = result["cluster_tsv"]
print(f"rep_fasta:   {rep_fasta}")
print(f"cluster_tsv: {cluster_tsv}")

13:22:41  INFO      CMD  mmseqs easy-cluster ../data/GH43_0.8-0.8_catdom.fa ../results/GH43_test/cluster/test_run ../results/GH43_test/cluster/tmp --min-seq-id 0.8 --cov-mode 0 -c 0.8
13:22:51  INFO           easy-cluster ../data/GH43_0.8-0.8_catdom.fa ../results/GH43_test/cluster/test_run ../results/GH43_test/cluster/tmp --min-seq-id 0.8 --cov-mode 0 -c 0.8 
13:22:51  INFO           
13:22:51  INFO           MMseqs Version:                     	17.b804f
13:22:51  INFO           Substitution matrix                 	aa:blosum62.out,nucl:nucleotide.out
13:22:51  INFO           Seed substitution matrix            	aa:VTML80.out,nucl:nucleotide.out
13:22:51  INFO           Sensitivity                         	4
13:22:51  INFO           k-mer length                        	0
13:22:51  INFO           Target search mode                  	0
13:22:51  INFO           k-score                             	seq:2147483647,prof:2147483647
13:22:51  INFO           Alphabet size                       	

rep_fasta:   ../results/GH43_test/cluster/test_run_rep_seq.fasta
cluster_tsv: ../results/GH43_test/cluster/test_run_cluster.tsv


### 1b · Skip clustering — use full input FASTA

In [ ]:
rep_fasta   = INPUT_FASTA
cluster_tsv = None
print(f"Using full input FASTA: {rep_fasta}")

---
## Step 2 — All-vs-all sequence similarity search (mmseqs)

Run **one** of the two cells below.

### 2a · Run mmseqs easy-search

In [4]:
from mmseqs import easy_search

result = easy_search(
    fasta       = rep_fasta,
    output_dir  = OUTPUT_DIR / "search",
    prefix      = PREFIX,
    evalue      = 1e-5,
    sensitivity = 7.5,
    log         = log,
)
m8_path = result["m8"]
print(f"m8: {m8_path}")

13:24:04  INFO      CMD  mmseqs easy-search ../results/GH43_test/cluster/test_run_rep_seq.fasta ../results/GH43_test/cluster/test_run_rep_seq.fasta ../results/GH43_test/search/test_run.m8 ../results/GH43_test/search/tmp --format-output query,target,pident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits -e 1e-05 -s 7.5
13:26:29  INFO           ../results/GH43_test/search/test_run.m8 exists and will be overwritten
13:26:29  INFO           easy-search ../results/GH43_test/cluster/test_run_rep_seq.fasta ../results/GH43_test/cluster/test_run_rep_seq.fasta ../results/GH43_test/search/test_run.m8 ../results/GH43_test/search/tmp --format-output query,target,pident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits -e 1e-05 -s 7.5 
13:26:29  INFO           
13:26:29  INFO           MMseqs Version:                        	17.b804f
13:26:29  INFO           Substitution matrix                    	aa:blosum62.out,nucl:nucleotide.out
13:26:29  INFO           Add backtrace       

m8: ../results/GH43_test/search/test_run.m8


### 2b · Use existing .m8 file

In [7]:
m8_path = Path("../results/GH43_0.8-0.8_catdom/search/test_run.m8")
print(f"Using existing m8: {m8_path}")

Using existing m8: ../results/GH43_0.8-0.8_catdom/search/test_run.m8


---
## Load similarity data

In [8]:
from plot_ssn import load_data

df = load_data(m8_path)
df.head()

13:40:20  INFO      START load similarity table ../results/GH43_0.8-0.8_catdom/search/test_run.m8
13:40:20  INFO      END   load similarity table ../results/GH43_0.8-0.8_catdom/search/test_run.m8: 0.16s
13:40:20  INFO      Loaded 463200 rows from ../results/GH43_0.8-0.8_catdom/search/test_run.m8
13:40:20  INFO      START remove self-hits
13:40:20  INFO      END   remove self-hits: 0.03s
13:40:20  INFO      Removed self-hits where qseqid == sseqid: 2570 rows
13:40:20  INFO      After removing self-hits: 460630 rows


Loaded data: 460630 edges


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
1,ABL84008.1,WKN48338.1,51.7,304,136,0,1,283,1,304,1.985000e-91,296
2,ABL84008.1,QIG44008.1,43.4,301,159,0,1,282,2,302,6.138000e-66,222
3,ABL84008.1,UDY24648.1,43.2,287,160,0,1,287,1,282,2.883000e-62,212
4,ABL84008.1,WQQ26163.1,42.9,301,160,0,1,282,3,303,1.379000e-61,210
5,ABL84008.1,QIK77284.1,38.2,303,174,0,1,282,1,303,2.813000e-59,203


---
## Load annotations *(optional)*

Skip this cell to produce uncolored networks.

In [4]:
import pandas as pd
from plot_ssn import build_color_lookup, _read_meta

META_FILE  = Path("../data/GH43_full.tsv")
ID_COL     = "GenBank"          # column matching sequence IDs in the m8 file
COLOR_COLS = ["Family", "Domain"]     # one plot is produced per column

meta = _read_meta(META_FILE, ID_COL)
print(f"Metadata: {len(meta)} rows  |  columns: {list(meta.columns)}")

color_lookups = {}
for col in COLOR_COLS:
    color_lookups[col] = build_color_lookup(meta, ID_COL, col, clusters=None)
    print(f"  '{col}': {len(color_lookups[col])} annotated entries")

Metadata: 63591 rows  |  columns: ['Family', 'Domain', 'Species', 'GenBank', 'Source']
  'Family': 61969 annotated entries
  'Domain': 61969 annotated entries


---
## Step 3a — Sequence Similarity Network (SSN)

In [11]:
import json
from plot_ssn import create_graph, exclude_singleton_components, get_layout, plot_ssn, graph_stats

THRESHOLDS         = [0.4, 0.6]       # pident thresholds (0–1)
NODE_SIZE          = 5
LAYOUT             = "component_grid" # "component_grid" | "fr" | "lgl" | "kk"
EXCLUDE_SINGLETONS = False

# Use color_lookups from the annotations cell, or {} for uncolored plots
_lookups    = locals().get("color_lookups", {})
_meta       = locals().get("meta", None)
_id_col     = locals().get("ID_COL", None)
_color_cols = list(_lookups.keys()) or [None]

(OUTPUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
ssn_stats = []

for thresh in THRESHOLDS:
    thresh_pident = thresh * 100 if thresh < 1 else thresh
    df_thresh = df[(df["pident"] >= thresh_pident) & (df["qseqid"] != df["sseqid"])].copy()

    g = create_graph(df_thresh)
    if g is None or g.vcount() == 0:
        print(f"t={thresh}: no nodes, skipping."); continue

    if EXCLUDE_SINGLETONS:
        g, _ = exclude_singleton_components(g)
        if g is None or g.vcount() == 0:
            print(f"t={thresh}: empty after singleton removal, skipping."); continue

    print(f"t={thresh}: {g.vcount()} nodes, {g.ecount()} edges")
    layout_coords = get_layout(g, LAYOUT)
    ssn_stats.append(graph_stats(g, thresh, metadata=_meta, id_col=_id_col))

    for col in _color_cols:
        lookup   = _lookups.get(col) or {n: "Unknown" for n in g.vs["name"]}
        col_lbl  = col if col else "uncolored"
        out_file = OUTPUT_DIR / "plots" / f"{PREFIX}_ssn_t{thresh}_{col_lbl}.html"
        plot_ssn(g, str(out_file), col, thresh, str(m8_path), lookup, NODE_SIZE,
                 layout_coords=layout_coords, metadata=_meta, id_col=_id_col)

stats_path = OUTPUT_DIR / "plots" / f"{PREFIX}_ssn_stats.json"
stats_path.write_text(json.dumps(ssn_stats, indent=2))
print(f"\nStats → {stats_path}")
pd.DataFrame(ssn_stats)

13:44:54  INFO      START deduplicate undirected edges
13:44:54  INFO      Deduplicated reciprocal/parallel edges: 86335 -> 45949
13:44:54  INFO      END   deduplicate undirected edges: 0.08s
13:44:54  INFO      START create edge list
13:44:54  INFO      END   create edge list: 0.01s
13:44:54  INFO      START build igraph
13:44:54  INFO      END   build igraph: 0.01s
13:44:54  INFO      START attach edge pident weights
13:44:54  INFO      END   attach edge pident weights: 0.00s
13:44:54  INFO      START compute component_grid layout (2480 nodes, 45949 edges)
13:44:55  INFO      END   compute component_grid layout (2480 nodes, 45949 edges): 0.66s


t=0.4: 2480 nodes, 45949 edges


13:44:55  INFO      START prepare node colors for Family
13:44:55  INFO      END   prepare node colors for Family: 0.00s
13:44:55  INFO      START build edge trace for Family
13:44:55  INFO      END   build edge trace for Family: 0.29s
13:44:55  INFO      START build node traces for Family (discrete)
13:45:00  INFO      END   build node traces for Family (discrete): 4.85s
13:45:00  INFO      START assemble figure for Family
13:45:00  INFO      END   assemble figure for Family: 0.30s
13:45:00  INFO      START write HTML ../results/GH43_test/plots/test_run_ssn_t0.4_Family.html
13:45:01  INFO      END   write HTML ../results/GH43_test/plots/test_run_ssn_t0.4_Family.html: 0.14s
13:45:01  INFO      Saved: ../results/GH43_test/plots/test_run_ssn_t0.4_Family.html
13:45:01  INFO      START write PNG ../results/GH43_test/plots/test_run_ssn_t0.4_Family.png


  Saved: ../results/GH43_test/plots/test_run_ssn_t0.4_Family.html


13:45:03  INFO      END   write PNG ../results/GH43_test/plots/test_run_ssn_t0.4_Family.png: 2.14s
13:45:03  INFO      Saved: ../results/GH43_test/plots/test_run_ssn_t0.4_Family.png
13:45:03  INFO      START prepare node colors for Domain
13:45:03  INFO      END   prepare node colors for Domain: 0.00s
13:45:03  INFO      START build edge trace for Domain


...

13:45:16  INFO      END   build node traces for Domain (discrete): 2.35s
13:45:16  INFO      START assemble figure for Domain
13:45:16  INFO      END   assemble figure for Domain: 0.02s
13:45:16  INFO      START write HTML ../results/GH43_test/plots/test_run_ssn_t0.6_Domain.html
13:45:16  INFO      END   write HTML ../results/GH43_test/plots/test_run_ssn_t0.6_Domain.html: 0.02s
13:45:16  INFO      Saved: ../results/GH43_test/plots/test_run_ssn_t0.6_Domain.html
13:45:16  INFO      START write PNG ../results/GH43_test/plots/test_run_ssn_t0.6_Domain.png


  Saved: ../results/GH43_test/plots/test_run_ssn_t0.6_Domain.html


13:45:16  INFO      END   write PNG ../results/GH43_test/plots/test_run_ssn_t0.6_Domain.png: 0.34s
13:45:16  INFO      Saved: ../results/GH43_test/plots/test_run_ssn_t0.6_Domain.png


  Saved: ../results/GH43_test/plots/test_run_ssn_t0.6_Domain.png

Stats → ../results/GH43_test/plots/test_run_ssn_stats.json


,threshold,nodes,edges,n_components,largest_component,singletons,annotated_nodes,annotated_percent
0,0.4,2480,45949,22,2369,0,2480,100.0
1,0.6,1205,1946,127,302,0,1205,100.0


---
## Step 3b — Minimum Spanning Tree / StSN *(optional)*

Reduces each connected component to its spanning tree (highest-similarity edges only).  
Useful for structure-based networks (foldseek) or for cleaner visualizations of dense SSNs.

In [12]:
from plot_mst import (
    build_mst_graph,
    exclude_singleton_components as mst_exclude_singletons,
    get_layout as mst_get_layout,
    plot_mst,
    graph_stats as mst_graph_stats,
)

THRESHOLDS         = [0.4, 0.6]       # pident thresholds (0–1)
NODE_SIZE          = 2
LAYOUT             = "component_grid"
EXCLUDE_SINGLETONS = True

_lookups    = locals().get("color_lookups", {})
_meta       = locals().get("meta", None)
_id_col     = locals().get("ID_COL", None)
_color_cols = list(_lookups.keys()) or [None]

(OUTPUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
mst_stats = []

for thresh in THRESHOLDS:
    thresh_pident = thresh * 100 if thresh < 1 else thresh
    df_thresh = df[(df["pident"] >= thresh_pident) & (df["qseqid"] != df["sseqid"])].copy()

    g = build_mst_graph(df_thresh)
    if g is None or g.vcount() == 0:
        print(f"t={thresh}: no nodes, skipping."); continue

    if EXCLUDE_SINGLETONS:
        g, _ = mst_exclude_singletons(g)
        if g is None or g.vcount() == 0:
            print(f"t={thresh}: empty after singleton removal, skipping."); continue

    print(f"t={thresh}: {g.vcount()} nodes, {g.ecount()} MST edges")
    layout_coords = mst_get_layout(g, LAYOUT)
    mst_stats.append(mst_graph_stats(g, thresh, metadata=_meta, id_col=_id_col))

    for col in _color_cols:
        lookup   = _lookups.get(col) or {n: "Unknown" for n in g.vs["name"]}
        col_lbl  = col if col else "uncolored"
        out_file = OUTPUT_DIR / "plots" / f"{PREFIX}_mst_t{thresh}_{col_lbl}.html"
        plot_mst(g, str(out_file), col, thresh, str(m8_path), lookup, NODE_SIZE,
                 layout_coords=layout_coords, metadata=_meta, id_col=_id_col)

stats_path = OUTPUT_DIR / "plots" / f"{PREFIX}_mst_stats.json"
stats_path.write_text(json.dumps(mst_stats, indent=2))
print(f"\nStats → {stats_path}")
pd.DataFrame(mst_stats)

13:46:14  INFO      START deduplicate undirected edges
13:46:14  INFO      Deduplicated edges: 86335 -> 45949
13:46:14  INFO      END   deduplicate undirected edges: 0.03s
13:46:14  INFO      START build node index
13:46:14  INFO      END   build node index: 0.00s
13:46:14  INFO      START build sparse distance matrix
13:46:14  INFO      END   build sparse distance matrix: 0.01s
13:46:14  INFO      START compute minimum spanning tree (forest)
13:46:14  INFO      END   compute minimum spanning tree (forest): 0.00s
13:46:14  INFO      START extract MST edges
13:46:14  INFO      MST has 2458 edges (from 45949 input edges)
13:46:14  INFO      END   extract MST edges: 0.00s
13:46:14  INFO      START build igraph from MST edges
13:46:14  INFO      END   build igraph from MST edges: 0.00s
13:46:14  INFO      START compute component_grid layout (2480 nodes, 2458 edges)
13:46:14  INFO      END   compute component_grid layout (2480 nodes, 2458 edges): 0.85s


t=0.4: 2480 nodes, 2458 MST edges


13:46:14  INFO      START prepare node colors for Family
13:46:14  INFO      END   prepare node colors for Family: 0.00s
13:46:14  INFO      START build edge trace for Family
13:46:14  INFO      END   build edge trace for Family: 0.02s
13:46:14  INFO      START build node traces for Family (discrete)
13:46:19  INFO      END   build node traces for Family (discrete): 4.96s
13:46:19  INFO      START assemble figure for Family
13:46:19  INFO      END   assemble figure for Family: 0.03s
13:46:19  INFO      START write HTML ../results/GH43_test/plots/test_run_mst_t0.4_Family.html
13:46:19  INFO      END   write HTML ../results/GH43_test/plots/test_run_mst_t0.4_Family.html: 0.02s
13:46:19  INFO      Saved: ../results/GH43_test/plots/test_run_mst_t0.4_Family.html
13:46:19  INFO      START write PNG ../results/GH43_test/plots/test_run_mst_t0.4_Family.png


  Saved: ../results/GH43_test/plots/test_run_mst_t0.4_Family.html


13:46:20  INFO      END   write PNG ../results/GH43_test/plots/test_run_mst_t0.4_Family.png: 0.37s
13:46:20  INFO      Saved: ../results/GH43_test/plots/test_run_mst_t0.4_Family.png
13:46:20  INFO      START prepare node colors for Domain
13:46:20  INFO      END   prepare node colors for Domain: 0.00s
13:46:20  INFO      START build edge trace for Domain
13:46:20  INFO      END   build edge trace for Domain: 0.02s
13:46:20  INFO      START build node traces for Domain (discrete)


...

13:46:28  INFO      END   write PNG ../results/GH43_test/plots/test_run_mst_t0.6_Family.png: 0.23s
13:46:28  INFO      Saved: ../results/GH43_test/plots/test_run_mst_t0.6_Family.png
13:46:28  INFO      START prepare node colors for Domain
13:46:28  INFO      END   prepare node colors for Domain: 0.00s
13:46:28  INFO      START build edge trace for Domain
13:46:28  INFO      END   build edge trace for Domain: 0.01s
13:46:28  INFO      START build node traces for Domain (discrete)


  Saved: ../results/GH43_test/plots/test_run_mst_t0.6_Family.png


13:46:30  INFO      END   build node traces for Domain (discrete): 2.36s
13:46:30  INFO      START assemble figure for Domain
13:46:30  INFO      END   assemble figure for Domain: 0.01s
13:46:30  INFO      START write HTML ../results/GH43_test/plots/test_run_mst_t0.6_Domain.html
13:46:30  INFO      END   write HTML ../results/GH43_test/plots/test_run_mst_t0.6_Domain.html: 0.02s
13:46:30  INFO      Saved: ../results/GH43_test/plots/test_run_mst_t0.6_Domain.html
13:46:30  INFO      START write PNG ../results/GH43_test/plots/test_run_mst_t0.6_Domain.png
13:46:30  INFO      END   write PNG ../results/GH43_test/plots/test_run_mst_t0.6_Domain.png: 0.17s
13:46:30  INFO      Saved: ../results/GH43_test/plots/test_run_mst_t0.6_Domain.png


  Saved: ../results/GH43_test/plots/test_run_mst_t0.6_Domain.html
  Saved: ../results/GH43_test/plots/test_run_mst_t0.6_Domain.png

Stats → ../results/GH43_test/plots/test_run_mst_stats.json


,threshold,nodes,mst_edges,n_components,largest_component,singletons,annotated_nodes,annotated_percent
0,0.4,2480,2458,22,2369,0,2480,100.0
1,0.6,1205,1078,127,302,0,1205,100.0


---
## Results

In [ ]:
from IPython.display import display, HTML

html_files = sorted((OUTPUT_DIR / "plots").glob("*.html"))
items = "".join(f'<li><a href="{f.resolve()}" target="_blank">{f.name}</a></li>' for f in html_files)
display(HTML(f"<ul>{items}</ul>") if items else HTML("<i>No plots found — run cells above first.</i>"))